# Mangrove Pulse: connected concept assessment

This notebook connects Khor Kalba review questions to method selection, necessary physical constraints and separate validation tracks. The drone remains concept-only; the calculations establish neither aircraft performance nor ecological outcomes.

Run all cells to check equations and record handling, display unknown engineering inputs and export a downloadable ZIP. The [original executed notebook](Mangrove_Pulse_Connected_Concept_Executed.ipynb) preserves the recorded run. This source copy has revised introductory prose; its code and embedded preparation note are unchanged.

## 1. Provenance and synthetic tests

The observation requests, screening module and tests are embedded below. The three cases contain 85 historical target references, not landing positions. Unit tests use artificial values to check equations and reject malformed records. They do not establish aircraft viability.

In [ ]:
SCREEN_SOURCE = "\"\"\"Conditional analytic screens; no flight simulation, accuracy or ecology claim.\"\"\"\nimport math\n\n\nPARAMETER_UNITS = {\n    \"all_up_wet_mass_kg\": \"kg\", \"water_density_kg_m3\": \"kg/m^3\",\n    \"displacement_limit_m3\": \"m^3\", \"gravity_m_s2\": \"m/s^2\",\n    \"available_thrust_n\": \"N\", \"tilt_deg\": \"degree\",\n    \"principal_inertia_kg_m2\": \"three kg*m^2 components\",\n    \"angular_velocity_rad_s\": \"three rad/s components\",\n    \"net_body_torque_nm\": \"three N*m components\",\n    \"usable_battery_wh\": \"Wh\", \"planned_complete_mission_wh\": \"Wh\",\n    \"additional_recovery_wh\": \"Wh\", \"required_reserve_wh\": \"Wh\",\n    \"available_peak_power_w\": \"W\", \"required_peak_power_w\": \"W\",\n    \"first_order_response_applicable\": \"boolean declaration\",\n    \"sensor_tau_s\": \"s\", \"stable_dwell_s\": \"s\",\n    \"residual_error_limit\": \"fraction of initial step error\",\n    \"hover_complete_energy_wh\": \"Wh\", \"supported_complete_energy_wh\": \"Wh\",\n    \"hover_complete_time_s\": \"s\", \"supported_complete_time_s\": \"s\",\n    \"hover_acceptance_fraction\": \"fraction\", \"supported_acceptance_fraction\": \"fraction\",\n}\n\n\ndef blank_parameters():\n    return {key: {\"value\": None, \"unit\": unit, \"kind\": \"unknown\",\n                  \"source\": None, \"conditions_and_uncertainty\": None}\n            for key, unit in PARAMETER_UNITS.items()}\n\n\ndef finite(value, name, minimum=None, maximum=None, strict_min=False):\n    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\n        raise ValueError(f\"{name}: finite number required\")\n    if minimum is not None and (value < minimum or (strict_min and value == minimum)):\n        raise ValueError(f\"{name}: outside lower bound\")\n    if maximum is not None and value > maximum:\n        raise ValueError(f\"{name}: outside upper bound\")\n    return value\n\n\ndef values_from_records(records):\n    if set(records) != set(PARAMETER_UNITS):\n        raise ValueError(\"Parameter keys must match the documented register\")\n    values = {}\n    for key, unit in PARAMETER_UNITS.items():\n        record = records[key]\n        if record.get(\"unit\") != unit:\n            raise ValueError(f\"{key}: wrong unit declaration\")\n        value = record.get(\"value\")\n        if value is None:\n            if record.get(\"kind\") != \"unknown\":\n                raise ValueError(f\"{key}: null must remain unknown\")\n        else:\n            if record.get(\"kind\") not in {\"assumption\", \"declared_measurement\", \"saved_hypothetical_result\"}:\n                raise ValueError(f\"{key}: evidence kind required\")\n            for field in (\"source\", \"conditions_and_uncertainty\"):\n                if not isinstance(record.get(field), str) or not record[field].strip():\n                    raise ValueError(f\"{key}: {field} required\")\n        values[key] = value\n    return values\n\n\ndef vector3(value, name, positive=False):\n    if not isinstance(value, (list, tuple)) or len(value) != 3:\n        raise ValueError(f\"{name}: three components required\")\n    return [finite(v, name, minimum=0 if positive else None, strict_min=positive) for v in value]\n\n\ndef euler_rhs(inertia, omega, torque):\n    \"\"\"Instantaneous derivative about CM in body principal axes, not stability.\"\"\"\n    ix, iy, iz = vector3(inertia, \"inertia\", positive=True)\n    largest = max(ix, iy, iz)\n    others = sorted((ix, iy, iz))[:2]\n    if largest / 2 > others[0] / 2 + others[1] / 2:\n        raise ValueError(\"Principal moments violate the physical inertia triangle\")\n    wx, wy, wz = vector3(omega, \"angular velocity\")\n    tx, ty, tz = vector3(torque, \"net body torque\")\n    result = [(tx + (iy - iz) * wy * wz) / ix,\n              (ty + (iz - ix) * wz * wx) / iy,\n              (tz + (ix - iy) * wx * wy) / iz]\n    return [finite(x, \"angular acceleration\") for x in result]\n\n\ndef margin_status(value):\n    finite(value, \"derived margin\")\n    return \"fails_necessary_screen\" if value < 0 else (\"boundary_no_margin\" if value == 0 else \"conditional_margin_only\")\n\n\ndef assess(records):\n    v = values_from_records(records)\n    # Validate supplied values even if another input in the same group is missing.\n    signed = {\"tilt_deg\", \"principal_inertia_kg_m2\", \"angular_velocity_rad_s\", \"net_body_torque_nm\", \"first_order_response_applicable\"}\n    zero_allowed = {\"additional_recovery_wh\", \"required_reserve_wh\", \"stable_dwell_s\", \"hover_acceptance_fraction\", \"supported_acceptance_fraction\"}\n    for key, value in v.items():\n        if value is not None and key not in signed:\n            finite(value, key, minimum=0, strict_min=key not in zero_allowed)\n    for key in (\"hover_acceptance_fraction\", \"supported_acceptance_fraction\", \"residual_error_limit\"):\n        if v[key] is not None:\n            finite(v[key], key, minimum=0, maximum=1, strict_min=key == \"residual_error_limit\")\n    if v[\"residual_error_limit\"] == 1:\n        raise ValueError(\"Residual error limit must be strictly less than one\")\n    if v[\"tilt_deg\"] is not None:\n        finite(v[\"tilt_deg\"], \"tilt\", minimum=0, maximum=89.999999)\n    for key in (\"principal_inertia_kg_m2\", \"angular_velocity_rad_s\", \"net_body_torque_nm\"):\n        if v[key] is not None:\n            vector3(v[key], key, positive=key == \"principal_inertia_kg_m2\")\n    if v[\"principal_inertia_kg_m2\"] is not None:\n        # Reject an impossible declared inertia even if velocities/torques are absent.\n        euler_rhs(v[\"principal_inertia_kg_m2\"], [0, 0, 0], [0, 0, 0])\n    if v[\"first_order_response_applicable\"] is not None and not isinstance(v[\"first_order_response_applicable\"], bool):\n        raise ValueError(\"Response applicability must be boolean or unknown\")\n    def ready(*keys):\n        return all(v[key] is not None for key in keys)\n    report = {\"status\": \"conditional_analytic_assessment_not_field_validation\",\n              \"deployment_authorized\": False,\n              \"claims\": {key: \"unproven\" for key in (\"independent_2018_accuracy\", \"drone_superiority\", \"ecological_benefit\")},\n              \"source_declarations_independently_verified\": False,\n              \"screens\": {key: {\"status\": \"unknown_missing_inputs\"} for key in\n                          (\"buoyancy\", \"static_vertical_thrust\", \"battery_energy\", \"peak_power\", \"sensor_settling\", \"euler_dynamics\", \"accepted_resource_frontier\")},\n              \"unassessed\": [\"task utility and acceptance rule\", \"independent dated reference truth\", \"permissions and habitat disturbance\",\n                             \"float restoring stability and freeboard\", \"wind waves current and station keeping\", \"water transitions and recovery\",\n                             \"closed-loop orientation and translational dynamics\", \"sensor bias calibration contamination and positioning\",\n                             \"ecological outcomes and lifecycle costs\"]}\n    s = report[\"screens\"]\n    if ready(\"all_up_wet_mass_kg\", \"water_density_kg_m3\", \"displacement_limit_m3\"):\n        required = finite(v[\"all_up_wet_mass_kg\"] / v[\"water_density_kg_m3\"], \"required displacement\")\n        margin = v[\"displacement_limit_m3\"] - required\n        s[\"buoyancy\"] = {\"status\": margin_status(margin), \"required_displacement_m3\": required, \"volume_margin_m3\": margin}\n    if ready(\"all_up_wet_mass_kg\", \"gravity_m_s2\", \"available_thrust_n\", \"tilt_deg\"):\n        margin = v[\"available_thrust_n\"] * math.cos(math.radians(v[\"tilt_deg\"])) - v[\"all_up_wet_mass_kg\"] * v[\"gravity_m_s2\"]\n        s[\"static_vertical_thrust\"] = {\"status\": margin_status(margin), \"vertical_margin_n\": margin}\n    if ready(\"usable_battery_wh\", \"planned_complete_mission_wh\", \"additional_recovery_wh\", \"required_reserve_wh\"):\n        margin = v[\"usable_battery_wh\"] - v[\"planned_complete_mission_wh\"] - v[\"additional_recovery_wh\"] - v[\"required_reserve_wh\"]\n        s[\"battery_energy\"] = {\"status\": margin_status(margin), \"remaining_after_all_costs_wh\": margin}\n    if ready(\"available_peak_power_w\", \"required_peak_power_w\"):\n        margin = v[\"available_peak_power_w\"] - v[\"required_peak_power_w\"]\n        s[\"peak_power\"] = {\"status\": margin_status(margin), \"power_margin_w\": margin}\n    if v[\"first_order_response_applicable\"] is False:\n        s[\"sensor_settling\"] = {\"status\": \"model_not_applicable\"}\n    elif v[\"first_order_response_applicable\"] is True and ready(\"sensor_tau_s\", \"stable_dwell_s\", \"residual_error_limit\"):\n        required = -v[\"sensor_tau_s\"] * math.log(v[\"residual_error_limit\"])\n        margin = v[\"stable_dwell_s\"] - required\n        s[\"sensor_settling\"] = {\"status\": margin_status(margin), \"required_stable_dwell_s\": finite(required, \"settling time\")}\n    if ready(\"principal_inertia_kg_m2\", \"angular_velocity_rad_s\", \"net_body_torque_nm\"):\n        s[\"euler_dynamics\"] = {\"status\": \"instantaneous_derivative_only_not_stability\", \"omega_dot_rad_s2\":\n                               euler_rhs(v[\"principal_inertia_kg_m2\"], v[\"angular_velocity_rad_s\"], v[\"net_body_torque_nm\"])}\n    if ready(\"hover_complete_energy_wh\", \"supported_complete_energy_wh\", \"hover_complete_time_s\", \"supported_complete_time_s\"):\n        er = finite(v[\"supported_complete_energy_wh\"] / v[\"hover_complete_energy_wh\"], \"energy ratio\")\n        tr = finite(v[\"supported_complete_time_s\"] / v[\"hover_complete_time_s\"], \"time ratio\")\n        f = {\"status\": \"conditional_thresholds_acceptance_unknown\", \"strict_energy_ratio_threshold\": er,\n             \"strict_time_ratio_threshold\": tr, \"strict_joint_ratio_threshold\": max(er, tr),\n             \"assumptions\": \"Identical attempted tasks and quality rules; complete expected costs across repeated missions, including failures/recovery\"}\n        if ready(\"hover_acceptance_fraction\", \"supported_acceptance_fraction\"):\n            ph, ps = v[\"hover_acceptance_fraction\"], v[\"supported_acceptance_fraction\"]\n            if ph == 0 or ps == 0:\n                f[\"status\"] = \"undefined_per_accepted_comparison_zero_acceptance\"\n            else:\n                ratio = finite(ps / ph, \"acceptance ratio\")\n                f.update(status=\"conditional_comparison_only\", declared_acceptance_ratio=ratio,\n                         energy_improves_under_inputs=ratio > er, time_improves_under_inputs=ratio > tr)\n        s[\"accepted_resource_frontier\"] = f\n    return report\n"
TEST_SOURCE = "\"\"\"Synthetic mathematical/metadata tests to execute in Colab, not field tests.\"\"\"\nimport unittest\nfrom concept_screen import assess, blank_parameters, euler_rhs\n\n\ndef supplied(**values):\n    records = blank_parameters()\n    for key, value in values.items():\n        records[key].update(value=value, kind=\"assumption\", source=\"synthetic unit test\",\n                            conditions_and_uncertainty=\"Artificial values, not any aircraft or site\")\n    return records\n\n\nclass ConceptTests(unittest.TestCase):\n    def test_unknown_is_not_success(self):\n        r = assess(blank_parameters())\n        self.assertTrue(all(x[\"status\"] == \"unknown_missing_inputs\" for x in r[\"screens\"].values()))\n        self.assertFalse(r[\"deployment_authorized\"])\n        self.assertTrue(all(x == \"unproven\" for x in r[\"claims\"].values()))\n\n    def test_buoyancy_failure(self):\n        r = assess(supplied(all_up_wet_mass_kg=10, water_density_kg_m3=1000, displacement_limit_m3=.009))\n        self.assertEqual(r[\"screens\"][\"buoyancy\"][\"status\"], \"fails_necessary_screen\")\n\n    def test_boundary_not_pass(self):\n        r = assess(supplied(all_up_wet_mass_kg=10, water_density_kg_m3=1000, displacement_limit_m3=.01))\n        self.assertEqual(r[\"screens\"][\"buoyancy\"][\"status\"], \"boundary_no_margin\")\n\n    def test_static_thrust(self):\n        r = assess(supplied(all_up_wet_mass_kg=1, gravity_m_s2=10, available_thrust_n=9, tilt_deg=0))\n        self.assertEqual(r[\"screens\"][\"static_vertical_thrust\"][\"status\"], \"fails_necessary_screen\")\n\n    def test_recovery_reserve_reverse_margin(self):\n        r = assess(supplied(usable_battery_wh=100, planned_complete_mission_wh=80, additional_recovery_wh=15, required_reserve_wh=10))\n        self.assertEqual(r[\"screens\"][\"battery_energy\"][\"remaining_after_all_costs_wh\"], -5)\n\n    def test_peak_power_separate_from_energy(self):\n        r = assess(supplied(available_peak_power_w=20, required_peak_power_w=30))\n        self.assertEqual(r[\"screens\"][\"peak_power\"][\"status\"], \"fails_necessary_screen\")\n\n    def test_euler_coupling_and_energy(self):\n        a = euler_rhs([2, 3, 4], [1, 2, 3], [0, 0, 0])\n        self.assertEqual(a, [-3, 2, -.5])\n        self.assertEqual(sum(i*w*x for i, w, x in zip([2, 3, 4], [1, 2, 3], a)), 0)\n\n    def test_euler_torque_at_rest(self):\n        self.assertEqual(euler_rhs([2, 3, 4], [0, 0, 0], [2, 6, 12]), [1, 2, 3])\n\n    def test_impossible_inertia(self):\n        with self.assertRaises(ValueError):\n            euler_rhs([1, 1, 3], [0, 0, 0], [0, 0, 0])\n        with self.assertRaises(ValueError):\n            assess(supplied(principal_inertia_kg_m2=[1, 1, 3]))\n\n    def test_response_requires_applicability(self):\n        r = assess(supplied(first_order_response_applicable=False, sensor_tau_s=1, stable_dwell_s=10, residual_error_limit=.05))\n        self.assertEqual(r[\"screens\"][\"sensor_settling\"][\"status\"], \"model_not_applicable\")\n\n    def test_settling_failure(self):\n        r = assess(supplied(first_order_response_applicable=True, sensor_tau_s=1, stable_dwell_s=1, residual_error_limit=.05))\n        self.assertEqual(r[\"screens\"][\"sensor_settling\"][\"status\"], \"fails_necessary_screen\")\n\n    def test_same_acceptance_different_tradeoffs(self):\n        r = assess(supplied(hover_complete_energy_wh=10, supported_complete_energy_wh=8, hover_complete_time_s=10,\n                           supported_complete_time_s=12, hover_acceptance_fraction=.5, supported_acceptance_fraction=.5))\n        f = r[\"screens\"][\"accepted_resource_frontier\"]\n        self.assertTrue(f[\"energy_improves_under_inputs\"])\n        self.assertFalse(f[\"time_improves_under_inputs\"])\n        self.assertFalse(r[\"deployment_authorized\"])\n\n    def test_zero_acceptance_not_division(self):\n        r = assess(supplied(hover_complete_energy_wh=10, supported_complete_energy_wh=8, hover_complete_time_s=10,\n                           supported_complete_time_s=12, hover_acceptance_fraction=0, supported_acceptance_fraction=.5))\n        self.assertEqual(r[\"screens\"][\"accepted_resource_frontier\"][\"status\"], \"undefined_per_accepted_comparison_zero_acceptance\")\n\n    def test_bad_partial_values(self):\n        for value in (True, float(\"nan\"), float(\"inf\"), -1, 0):\n            with self.subTest(value=value), self.assertRaises(ValueError):\n                assess(supplied(all_up_wet_mass_kg=value))\n\n    def test_missing_provenance_or_wrong_unit(self):\n        for field, value in ((\"source\", \"\"), (\"unit\", \"litres\")):\n            r = supplied(displacement_limit_m3=.01)\n            r[\"displacement_limit_m3\"][field] = value\n            with self.assertRaises(ValueError):\n                assess(r)\n\n    def test_unknown_recovery_not_zero(self):\n        r = assess(supplied(usable_battery_wh=100, planned_complete_mission_wh=80, required_reserve_wh=10))\n        self.assertEqual(r[\"screens\"][\"battery_energy\"][\"status\"], \"unknown_missing_inputs\")\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n"
REQUESTS_TEXT = "{\n  \"schema_version\": \"1.0\",\n  \"package_id\": \"MP-KHOR-KALBA-HANDOFF-01\",\n  \"prepared_date\": \"2026-10-07\",\n  \"evidence_status\": \"Cached historical evidence and proposed observation requirements; no inspection performed\",\n  \"coordinate_semantics\": \"WGS84 original diagnostic cell centres, not route, waypoint, access, landing or contact sampling positions\",\n  \"historical_satellite_dates\": [\n    \"2018-11-20\",\n    \"2018-11-30\"\n  ],\n  \"current_field_access\": \"unconfirmed\",\n  \"sources\": {\n    \"../audit/all_audit_cells.csv\": \"80a92cf778a92f8bdcc4a99e39dfe906f193a3eaad1dacbbd166f13481806e73\",\n    \"../reference_context/case_comparison.json\": \"895116d01d0c79ce8191a63de503051304f58c32fe32238061ab962b92492f35\"\n  },\n  \"temporal_gate\": \"Current observations cannot directly validate 2018 cover; record actual dates and preserve unresolved historical truth\",\n  \"cases\": [\n    {\n      \"case_id\": \"C001\",\n      \"target_count\": 11,\n      \"original_category\": \"Prediction positive / GMW not mapped\",\n      \"cell_ids\": [\n        \"C001_r88_c83\",\n        \"C001_r89_c83\",\n        \"C001_r90_c82\",\n        \"C001_r91_c81\",\n        \"C001_r92_c80\",\n        \"C001_r92_c81\",\n        \"C001_r93_c80\",\n        \"C001_r93_c81\",\n        \"C001_r94_c81\",\n        \"C001_r95_c81\",\n        \"C001_r96_c82\"\n      ],\n      \"reference_comparison_2018\": {\n        \"both_not_mapped\": 4,\n        \"v3_only_mapped\": 0,\n        \"v4_only_mapped\": 7,\n        \"both_mapped\": 0,\n        \"centres\": 11,\n        \"differently_mapped_centres\": 7,\n        \"differently_mapped_fraction\": 0.6363636363636364,\n        \"mapped_jaccard\": 0,\n        \"interpretation\": \"Reference-map agreement, not field accuracy or ecological change\"\n      },\n      \"worldcover_2020_later_context\": [\n        {\n          \"code\": 95,\n          \"class\": \"Mangroves\",\n          \"centres\": 11\n        }\n      ],\n      \"cell_truth\": \"unresolved\",\n      \"question\": \"Which target cells and surroundings contain identifiable canopy at the observation date, and how certain is the edge alignment?\",\n      \"request\": [\n        \"Dated aligned cover evidence across cells and surrounding edge\",\n        \"Qualified vegetation identification\",\n        \"Visibility, occlusion and positioning uncertainty\"\n      ],\n      \"candidate_method\": \"Existing licensed imagery or safe shore/boat photography; ordinary RGB drone only if access and observation quality justify it\",\n      \"amphibious_need\": \"Not established for photographic canopy-edge review\",\n      \"deployment_status\": \"not_authorized_or_planned\",\n      \"field_observations_collected\": 0,\n      \"planting_recommendation\": \"none\"\n    },\n    {\n      \"case_id\": \"C017\",\n      \"target_count\": 32,\n      \"original_category\": \"Prediction negative / GMW mapped\",\n      \"cell_ids\": [\n        \"C017_r64_c87\",\n        \"C017_r64_c88\",\n        \"C017_r64_c89\",\n        \"C017_r65_c86\",\n        \"C017_r65_c87\",\n        \"C017_r65_c88\",\n        \"C017_r65_c89\",\n        \"C017_r65_c90\",\n        \"C017_r66_c84\",\n        \"C017_r66_c86\",\n        \"C017_r66_c87\",\n        \"C017_r66_c88\",\n        \"C017_r67_c84\",\n        \"C017_r67_c85\",\n        \"C017_r67_c86\",\n        \"C017_r67_c87\",\n        \"C017_r68_c84\",\n        \"C017_r68_c85\",\n        \"C017_r68_c86\",\n        \"C017_r68_c87\",\n        \"C017_r69_c84\",\n        \"C017_r69_c85\",\n        \"C017_r69_c86\",\n        \"C017_r69_c87\",\n        \"C017_r70_c84\",\n        \"C017_r70_c85\",\n        \"C017_r71_c85\",\n        \"C017_r72_c85\",\n        \"C017_r73_c85\",\n        \"C017_r74_c83\",\n        \"C017_r74_c84\",\n        \"C017_r75_c84\"\n      ],\n      \"reference_comparison_2018\": {\n        \"both_not_mapped\": 0,\n        \"v3_only_mapped\": 26,\n        \"v4_only_mapped\": 0,\n        \"both_mapped\": 6,\n        \"centres\": 32,\n        \"differently_mapped_centres\": 26,\n        \"differently_mapped_fraction\": 0.8125,\n        \"mapped_jaccard\": 0.1875,\n        \"interpretation\": \"Reference-map agreement, not field accuracy or ecological change\"\n      },\n      \"worldcover_2020_later_context\": [\n        {\n          \"code\": 60,\n          \"class\": \"Bare / sparse vegetation\",\n          \"centres\": 7\n        },\n        {\n          \"code\": 90,\n          \"class\": \"Herbaceous wetland\",\n          \"centres\": 4\n        },\n        {\n          \"code\": 95,\n          \"class\": \"Mangroves\",\n          \"centres\": 21\n        }\n      ],\n      \"cell_truth\": \"unresolved\",\n      \"question\": \"What cover composition can be supported for each cell, including mixed vegetation, water and open ground?\",\n      \"request\": [\n        \"Cover categories with uncertainty\",\n        \"Canopy/open-cover fractions only with adequate aligned coverage and documented labelling method\",\n        \"Qualified vegetation identification\"\n      ],\n      \"candidate_method\": \"Detailed licensed imagery, shore/boat photography or ordinary RGB drone where justified\",\n      \"amphibious_need\": \"Not established for photographic mixed-cover review\",\n      \"deployment_status\": \"not_authorized_or_planned\",\n      \"field_observations_collected\": 0,\n      \"planting_recommendation\": \"none\"\n    },\n    {\n      \"case_id\": \"C027\",\n      \"target_count\": 42,\n      \"original_category\": \"Quality-excluded / GMW mapped\",\n      \"cell_ids\": [\n        \"C027_r61_c96\",\n        \"C027_r62_c96\",\n        \"C027_r63_c96\",\n        \"C027_r64_c96\",\n        \"C027_r64_c97\",\n        \"C027_r65_c95\",\n        \"C027_r65_c97\",\n        \"C027_r66_c96\",\n        \"C027_r66_c97\",\n        \"C027_r67_c96\",\n        \"C027_r67_c97\",\n        \"C027_r68_c96\",\n        \"C027_r68_c97\",\n        \"C027_r69_c96\",\n        \"C027_r69_c97\",\n        \"C027_r70_c96\",\n        \"C027_r70_c97\",\n        \"C027_r71_c96\",\n        \"C027_r72_c95\",\n        \"C027_r72_c96\",\n        \"C027_r73_c95\",\n        \"C027_r73_c96\",\n        \"C027_r74_c95\",\n        \"C027_r74_c96\",\n        \"C027_r75_c96\",\n        \"C027_r76_c96\",\n        \"C027_r76_c97\",\n        \"C027_r77_c96\",\n        \"C027_r77_c97\",\n        \"C027_r78_c97\",\n        \"C027_r78_c98\",\n        \"C027_r79_c97\",\n        \"C027_r79_c98\",\n        \"C027_r80_c97\",\n        \"C027_r80_c98\",\n        \"C027_r80_c99\",\n        \"C027_r81_c98\",\n        \"C027_r81_c99\",\n        \"C027_r82_c99\",\n        \"C027_r82_c100\",\n        \"C027_r83_c100\",\n        \"C027_r84_c100\"\n      ],\n      \"reference_comparison_2018\": {\n        \"both_not_mapped\": 0,\n        \"v3_only_mapped\": 6,\n        \"v4_only_mapped\": 0,\n        \"both_mapped\": 36,\n        \"centres\": 42,\n        \"differently_mapped_centres\": 6,\n        \"differently_mapped_fraction\": 0.14285714285714285,\n        \"mapped_jaccard\": 0.8571428571428571,\n        \"interpretation\": \"Reference-map agreement, not field accuracy or ecological change\"\n      },\n      \"worldcover_2020_later_context\": [\n        {\n          \"code\": 10,\n          \"class\": \"Tree cover\",\n          \"centres\": 2\n        },\n        {\n          \"code\": 20,\n          \"class\": \"Shrubland\",\n          \"centres\": 3\n        },\n        {\n          \"code\": 80,\n          \"class\": \"Permanent water\",\n          \"centres\": 9\n        },\n        {\n          \"code\": 95,\n          \"class\": \"Mangroves\",\n          \"centres\": 28\n        }\n      ],\n      \"cell_truth\": \"unresolved\",\n      \"question\": \"What dated cover evidence is available at the channel margin, and what limits the original usable satellite coverage?\",\n      \"request\": [\n        \"Historical per-cell quality/input review kept separate from physical causes\",\n        \"Dated aligned vegetation/water/open-cover observations\",\n        \"Visibility and safe-access assessment\"\n      ],\n      \"candidate_method\": \"Historical source review first; current channel-edge photography only as a separate workflow trial\",\n      \"amphibious_need\": \"Conditional separate current contact-sensing hypothesis, not inferred from exclusions\",\n      \"conditional_contact_task\": \"If a qualified practitioner establishes a water-contact question: conductivity/salinity, temperature, depth and calibration information at separately justified safe sampling sites; no measurements or instrument validation exist\",\n      \"deployment_status\": \"not_authorized_or_planned\",\n      \"field_observations_collected\": 0,\n      \"planting_recommendation\": \"none\"\n    }\n  ]\n}\n"
NOTE_TEXT = "# Connected concept assessment: observation first, aircraft second\n\n7 October 2026 | Proposed engineering assessment, not a prototype or deployment plan\n\n## The complete connection\n\nHistorical satellite evidence -> unresolved question -> dated observation contract -> simplest suitable method -> physical and measurement checks -> accepted evidence -> qualified decision -> separately tested ecological outcome.\n\nThe completed PoC supplies the first two links and a traceable [observation handoff](DRONE_KHOR_KALBA_HANDOFF.md). It does not supply flight performance, independent 2018 truth or ecological outcomes. The team reconfirmed on 7 October that the drone is still concept-only. Mathematics can expose necessary conditions and reject inconsistent assumptions; it cannot fill the missing observations.\n\n| Evidence branch | Decision-relevant observation | Appropriate first comparison | Amphibious connection |\n|---|---|---|---|\n| C001, 11 original centres | Dated identifiable canopy and edge alignment | Licensed near-date imagery for 2018; separately, shore/boat imagery or ordinary RGB aerial imagery for current cover | Contact sensing does not answer the photographic question. No amphibious requirement established. |\n| C017, 32 centres | Mixed-cover composition, visibility and qualified interpretation | The same photographic alternatives, with adequate coverage of the whole cell and surroundings | A spectral or water reading is not a cover fraction or species label. |\n| C027, 42 centres | Historical optical quality/cover uncertainty; separately, a justified current channel observation | Source-quality review first; current photography if warranted. For a separately justified contact task, compare shore/boat probe, USV and hovering or floating platforms | Sustained contact is a candidate use, not a consequence of the exclusion flag. Sampling positions require a fresh site assessment. |\n\nThe 85 centres are historical references, not sampling stations, routes or landing coordinates. These selected cases are not an accuracy sample. A current visit cannot directly validate the 2018 map. [Site-specific published context](KHOR_KALBA_SITE_EVIDENCE.md) confirms older drone mapping and sediment sampling at Khor Kalba, not the performance of our concept.\n\n## What must enter the assessment\n\nThese are consequential parameter families, not a claim that every environmental interaction is modelled. Each supplied value needs units, provenance, date/conditions and an uncertainty range. Unknown is a valid result. A literature value or assumed design value is not a team measurement.\n\n| Family | Required information | What changes the conclusion |\n|---|---|---|\n| Observation and dates | Intended decision, observable, historical/current date, acceptance rule, independent reference lineage | A useful current observation can still be invalid for historical accuracy. No information benefit is established without a decision and acceptance rule. |\n| Access and habitat | Permissions, protected habitats, seasonal wildlife, launch/retrieval, qualified operator, lawful flight/contact constraints | A resource advantage cannot override a habitat or access constraint. Neither coordinates nor this assessment authorize deployment. |\n| Mass and flight | All-up wet mass, payload/float/battery mass, air density, thrust at conditions, rotor/actuator limits, peak power, centre of mass | A heavier float kit can erase energy savings or exceed lift/battery limits. Payload must fit the same aircraft budget. |\n| Water support and dynamics | Displacement/freeboard, hull geometry, centre of buoyancy, restoring stability, inertia, wind/waves/current, wet takeoff, net torque, drag, tether | Enough buoyancy does not imply upright stability, station keeping or recoverability. A tether can add forces and contaminate the measurement. |\n| Sensing and optical quality | Sensor calibration/response, depth, tide, positioning/alignment, coverage/occlusion, GSD, illumination, vibration, contamination and craft-induced disturbance | A longer dwell helps only if it yields an accepted measurement; rotor wash, stirred sediment or probe drift can bias it. Conductivity is not automatically calibrated salinity. |\n| Whole mission and reliability | Transit and all transitions, sampling, electronics, retries, recovery, battery derating, reserve, operator effort, failed/zero-success attempts | Compare complete costs, not motor-off dwell alone. Failures and recovery can reverse a nominal advantage. |\n| Navigation and links | Actual positioning error, antenna height, line of sight, logged packet delivery, loss-of-link behaviour, onboard recording and safe recovery | Satellite target coordinates are not measured aircraft positions; a relay aircraft adds its own energy, operator and disturbance costs. |\n| Ecology and economics | Baseline habitat/recruitment, hydrology/sediment, alternatives including no action, repeated controls, disturbance, cost, maintenance and lifecycle | More measurements or planted seedlings are not automatically better ecological outcomes. Carbon needs separate accounting. |\n\nHyperspectral, mud mobility, planting, solar, fleet coordination and communications are separate hypotheses. Add each only if it solves a defined observation or operational requirement and survives a matched simpler baseline. No covering hyperspectral experiment or implemented payload is claimed.\n\n## Mathematics that is useful now\n\nA [2024 Japanese pond study](https://mdpi-res.com/d_attachment/drones/drones-08-00038/article_deploy/drones-08-00038.pdf) tested amphibious sensing and relay links. Floats impaired flight; vegetation disrupted communications; an oxygen-consuming DO electrode made longer settling inappropriate in its test. Data are available by author request, not an established downloaded dataset. This supports selected component feasibility, not our design, Khor Kalba transfer or matched energy superiority.\n\n### Buoyancy and static lift: necessary, not sufficient\n\nFor a freely floating body at rest without other vertical forces, buoyancy balances weight: `rho_water * g * V_displaced = m * g`, hence `V_required = m / rho_water`. Compare this with a documented displacement limit that preserves the intended freeboard. Geometry, leaks, loading and restoring stability still require separate assessment. This follows [Archimedes' principle](https://openstax.org/books/university-physics-volume-1/pages/14-4-archimedes-principle-and-buoyancy).\n\nFor an idealized steady airborne state with no other vertical force, `T_available * cos(tilt) >= m * g`. Equality gives no upward thrust margin. This is a necessary static screen, not proof of controllability, flight endurance or wet takeoff. Use thrust and battery capability under the actual anticipated conditions, not an unrelated catalogue maximum.\n\n### Euler: where it genuinely belongs\n\nIn constant-mass rigid-body principal axes, the forced Euler equation is:\n\n`I * omega_dot + omega cross (I * omega) = torque_net`.\n\nFor example, `omega_dot_x = (torque_x + (I_y - I_z) * omega_y * omega_z) / I_x`, with cyclic counterparts. Angular velocity is in rad/s, inertia in kg m² and torque in N m. The [MIT Press mechanics text, section 2.9](https://mitp-content-server.mit.edu/books/content/sectbyfn/books_pres_0/9579/sicm_edition_2.zip/chapter002.html) derives the forced equation and distinguishes angular dynamics from orientation kinematics.\n\nThis identifies the inertia and torque inputs needed for contact/transition attitude assessment. An instantaneous acceleration is **not** a stability result. A future model needs orientation, translational dynamics, controller/actuator limits, hydrodynamic forces and restoring torques, and measured validation. Moving payloads, rotor angular momentum, variable mass and added-water effects require an expanded model or justified approximation. No such calibrated model exists here. Euler's differential equations and a numerical Euler integration method are different things; selecting an integrator does not validate the physics.\n\n### Dwell, measurement quality and resource advantage\n\nThe optional notebook includes a declared first-order response assumption: if a sensor's normalized step-error obeys `e(t) = exp(-t / tau)`, a residual error limit `epsilon` requires `t >= -tau * ln(epsilon)`. This is the team's elementary model derivation, not a specification for any selected sensor. Test applicability; settling alone excludes calibration bias, fouling and spatial drift.\n\nFor the same `n` attempted observation tasks and acceptance fraction `p > 0`, long-run expected energy per accepted observation is `E / (n*p)` and time is `T / (n*p)`. These are ratios of expectations, not the expected per-mission ratio; a mission can return zero accepted records. With equal attempted tasks, supported mode improves energy only if `p_S/p_H > E_S/E_H`, and time only if `p_S/p_H > T_S/T_H`. Acceptance must use the same predeclared quality rule, not different standards for each platform.\n\nThe [saved illustrative scenario](../examples/inspection/energy_scenario_excerpt.json) already shows approximately 170.7 versus 128.0 Wh but 750 versus 840 s. Those rounded hypothetical results imply an energy threshold near 0.749854 and a time threshold of 1.12; equal acceptance cannot improve both. A 30 s dwell or extra recovery overhead can reverse the energy advantage. These are not measured probabilities or UAE product comparisons. Unmeasured recovery costs must not silently become zero.\n\n## Choosing the next experiment, not the most elaborate aircraft\n\nThe outside-the-box contribution is **method selection tied to an evidence gap**, rather than assigning every satellite flag to the same drone. First ask which observation could change a decision and whether an existing source can answer it. Do not infer a probability of damage from disagreement frequency. Compare alternatives using information quality, habitat safeguards, energy, time, operator effort and money separately; do not hide trade-offs in an arbitrary weighted score.\n\nFor engineering, the smallest decisive future experiment is a permitted controlled-water comparison of the same specified sensor observation: establish calibration, depth/position/settling and bias against a reference probe; then measure complete energy, elapsed time, rejected attempts, transitions and recovery against the simplest suitable method. Include a no-amphibious baseline. Tank performance would be an early engineering result, not Khor Kalba field suitability or ecological benefit. No tank test is claimed or scheduled.\n\nFor accuracy, first establish reusable independent sufficiently detailed near-date evidence, then draw the frozen held-out probability sample and perform blind interpretation. For ecology, first establish current need and suitability, with no action/protection/natural regeneration as genuine alternatives; any intervention requires the [separate outcome protocol](VALIDATION_AND_BENEFIT_TESTS.md). The three evidence tracks cannot substitute for each other.\n\n## Reproducible companion and status\n\n[Connected concept Colab notebook](../supplementary/Mangrove_Pulse_Connected_Concept.ipynb) embeds the unchanged saved requests, this note, exact screening code/tests and blank parameter records. It needs no dataset upload, package installation or accelerator, and exports a downloadable assessment ZIP when run in Colab. Source is [concept_screen.py](../src/concept_screen.py). Default engineering inputs are unknown; saved energy/time values remain explicitly hypothetical. It never returns deployment authorization, independent accuracy, superiority or ecological-benefit proof. Synthetic tests test equations and record handling only.\n\nSource prepared; new Colab execution is not yet verified. The frozen EO inputs, threshold, split, predictions, scores and actual previously executed notebooks are unchanged. No new field data, local scientific run, PDF, publication or submission.\n"
REQUESTS_SHA256 = "3bceb31f3763f3690bc661b7d110d9efd7cde9493e267b8149bd76ca5060a59d"

import hashlib
import json
import sys
import types
import unittest

assert hashlib.sha256(REQUESTS_TEXT.encode('utf-8')).hexdigest() == REQUESTS_SHA256
requests = json.loads(REQUESTS_TEXT)
assert [(c['case_id'], c['target_count']) for c in requests['cases']] == [('C001', 11), ('C017', 32), ('C027', 42)]
assert all(c['field_observations_collected'] == 0 for c in requests['cases'])
assert sum(len(c['cell_ids']) for c in requests['cases']) == 85
module = types.ModuleType('concept_screen')
exec(SCREEN_SOURCE, module.__dict__)
sys.modules['concept_screen'] = module
test_module = types.ModuleType('test_concept_screen')
exec(TEST_SOURCE, test_module.__dict__)
result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromModule(test_module))
assert result.wasSuccessful(), 'Synthetic tests failed'
print('Synthetic equation and record tests passed; no field-validation claim.')
for case in requests['cases']:
    print(case['case_id'], case['target_count'], case['question'])


## 2. Archived reasoning and sources

The preparation note below is preserved with the embedded calculation inputs. Its execution-status statements describe preparation, not the later recorded run. See the [current connected assessment](../docs/CONNECTED_CONCEPT_ASSESSMENT.md) for subsequent source qualification and interpretation.

# Connected concept assessment: observation first, aircraft second

7 October 2026 | Proposed engineering assessment, not a prototype or deployment plan

## The complete connection

Historical satellite evidence -> unresolved question -> dated observation contract -> simplest suitable method -> physical and measurement checks -> accepted evidence -> qualified decision -> separately tested ecological outcome.

The completed PoC supplies the first two links and a traceable observation handoff (repository path: `DRONE_KHOR_KALBA_HANDOFF.md`). It does not supply flight performance, independent 2018 truth or ecological outcomes. The team reconfirmed on 7 October that the drone is still concept-only. Mathematics can expose necessary conditions and reject inconsistent assumptions; it cannot fill the missing observations.

| Evidence branch | Decision-relevant observation | Appropriate first comparison | Amphibious connection |
|---|---|---|---|
| C001, 11 original centres | Dated identifiable canopy and edge alignment | Licensed near-date imagery for 2018; separately, shore/boat imagery or ordinary RGB aerial imagery for current cover | Contact sensing does not answer the photographic question. No amphibious requirement established. |
| C017, 32 centres | Mixed-cover composition, visibility and qualified interpretation | The same photographic alternatives, with adequate coverage of the whole cell and surroundings | A spectral or water reading is not a cover fraction or species label. |
| C027, 42 centres | Historical optical quality/cover uncertainty; separately, a justified current channel observation | Source-quality review first; current photography if warranted. For a separately justified contact task, compare shore/boat probe, USV and hovering or floating platforms | Sustained contact is a candidate use, not a consequence of the exclusion flag. Sampling positions require a fresh site assessment. |

The 85 centres are historical references, not sampling stations, routes or landing coordinates. These selected cases are not an accuracy sample. A current visit cannot directly validate the 2018 map. Site-specific published context (repository path: `KHOR_KALBA_SITE_EVIDENCE.md`) confirms older drone mapping and sediment sampling at Khor Kalba, not the performance of our concept.

## What must enter the assessment

These are consequential parameter families, not a claim that every environmental interaction is modelled. Each supplied value needs units, provenance, date/conditions and an uncertainty range. Unknown is a valid result. A literature value or assumed design value is not a team measurement.

| Family | Required information | What changes the conclusion |
|---|---|---|
| Observation and dates | Intended decision, observable, historical/current date, acceptance rule, independent reference lineage | A useful current observation can still be invalid for historical accuracy. No information benefit is established without a decision and acceptance rule. |
| Access and habitat | Permissions, protected habitats, seasonal wildlife, launch/retrieval, qualified operator, lawful flight/contact constraints | A resource advantage cannot override a habitat or access constraint. Neither coordinates nor this assessment authorize deployment. |
| Mass and flight | All-up wet mass, payload/float/battery mass, air density, thrust at conditions, rotor/actuator limits, peak power, centre of mass | A heavier float kit can erase energy savings or exceed lift/battery limits. Payload must fit the same aircraft budget. |
| Water support and dynamics | Displacement/freeboard, hull geometry, centre of buoyancy, restoring stability, inertia, wind/waves/current, wet takeoff, net torque, drag, tether | Enough buoyancy does not imply upright stability, station keeping or recoverability. A tether can add forces and contaminate the measurement. |
| Sensing and optical quality | Sensor calibration/response, depth, tide, positioning/alignment, coverage/occlusion, GSD, illumination, vibration, contamination and craft-induced disturbance | A longer dwell helps only if it yields an accepted measurement; rotor wash, stirred sediment or probe drift can bias it. Conductivity is not automatically calibrated salinity. |
| Whole mission and reliability | Transit and all transitions, sampling, electronics, retries, recovery, battery derating, reserve, operator effort, failed/zero-success attempts | Compare complete costs, not motor-off dwell alone. Failures and recovery can reverse a nominal advantage. |
| Navigation and links | Actual positioning error, antenna height, line of sight, logged packet delivery, loss-of-link behaviour, onboard recording and safe recovery | Satellite target coordinates are not measured aircraft positions; a relay aircraft adds its own energy, operator and disturbance costs. |
| Ecology and economics | Baseline habitat/recruitment, hydrology/sediment, alternatives including no action, repeated controls, disturbance, cost, maintenance and lifecycle | More measurements or planted seedlings are not automatically better ecological outcomes. Carbon needs separate accounting. |

Hyperspectral, mud mobility, planting, solar, fleet coordination and communications are separate hypotheses. Add each only if it solves a defined observation or operational requirement and survives a matched simpler baseline. No covering hyperspectral experiment or implemented payload is claimed.

## Mathematics that is useful now

A [2024 Japanese pond study](https://mdpi-res.com/d_attachment/drones/drones-08-00038/article_deploy/drones-08-00038.pdf) tested amphibious sensing and relay links. Floats impaired flight; vegetation disrupted communications; an oxygen-consuming DO electrode made longer settling inappropriate in its test. Data are available by author request, not an established downloaded dataset. This supports selected component feasibility, not our design, Khor Kalba transfer or matched energy superiority.

### Buoyancy and static lift: necessary, not sufficient

For a freely floating body at rest without other vertical forces, buoyancy balances weight: `rho_water * g * V_displaced = m * g`, hence `V_required = m / rho_water`. Compare this with a documented displacement limit that preserves the intended freeboard. Geometry, leaks, loading and restoring stability still require separate assessment. This follows [Archimedes' principle](https://openstax.org/books/university-physics-volume-1/pages/14-4-archimedes-principle-and-buoyancy).

For an idealized steady airborne state with no other vertical force, `T_available * cos(tilt) >= m * g`. Equality gives no upward thrust margin. This is a necessary static screen, not proof of controllability, flight endurance or wet takeoff. Use thrust and battery capability under the actual anticipated conditions, not an unrelated catalogue maximum.

### Euler: where it genuinely belongs

In constant-mass rigid-body principal axes, the forced Euler equation is:

`I * omega_dot + omega cross (I * omega) = torque_net`.

For example, `omega_dot_x = (torque_x + (I_y - I_z) * omega_y * omega_z) / I_x`, with cyclic counterparts. Angular velocity is in rad/s, inertia in kg m² and torque in N m. The [MIT Press mechanics text, section 2.9](https://mitp-content-server.mit.edu/books/content/sectbyfn/books_pres_0/9579/sicm_edition_2.zip/chapter002.html) derives the forced equation and distinguishes angular dynamics from orientation kinematics.

This identifies the inertia and torque inputs needed for contact/transition attitude assessment. An instantaneous acceleration is **not** a stability result. A future model needs orientation, translational dynamics, controller/actuator limits, hydrodynamic forces and restoring torques, and measured validation. Moving payloads, rotor angular momentum, variable mass and added-water effects require an expanded model or justified approximation. No such calibrated model exists here. Euler's differential equations and a numerical Euler integration method are different things; selecting an integrator does not validate the physics.

### Dwell, measurement quality and resource advantage

The optional notebook includes a declared first-order response assumption: if a sensor's normalized step-error obeys `e(t) = exp(-t / tau)`, a residual error limit `epsilon` requires `t >= -tau * ln(epsilon)`. This is the team's elementary model derivation, not a specification for any selected sensor. Test applicability; settling alone excludes calibration bias, fouling and spatial drift.

For the same `n` attempted observation tasks and acceptance fraction `p > 0`, long-run expected energy per accepted observation is `E / (n*p)` and time is `T / (n*p)`. These are ratios of expectations, not the expected per-mission ratio; a mission can return zero accepted records. With equal attempted tasks, supported mode improves energy only if `p_S/p_H > E_S/E_H`, and time only if `p_S/p_H > T_S/T_H`. Acceptance must use the same predeclared quality rule, not different standards for each platform.

The saved illustrative scenario (repository path: `../examples/inspection/energy_scenario_excerpt.json`) already shows approximately 170.7 versus 128.0 Wh but 750 versus 840 s. Those rounded hypothetical results imply an energy threshold near 0.749854 and a time threshold of 1.12; equal acceptance cannot improve both. A 30 s dwell or extra recovery overhead can reverse the energy advantage. These are not measured probabilities or UAE product comparisons. Unmeasured recovery costs must not silently become zero.

## Choosing the next experiment, not the most elaborate aircraft

The outside-the-box contribution is **method selection tied to an evidence gap**, rather than assigning every satellite flag to the same drone. First ask which observation could change a decision and whether an existing source can answer it. Do not infer a probability of damage from disagreement frequency. Compare alternatives using information quality, habitat safeguards, energy, time, operator effort and money separately; do not hide trade-offs in an arbitrary weighted score.

For engineering, the smallest decisive future experiment is a permitted controlled-water comparison of the same specified sensor observation: establish calibration, depth/position/settling and bias against a reference probe; then measure complete energy, elapsed time, rejected attempts, transitions and recovery against the simplest suitable method. Include a no-amphibious baseline. Tank performance would be an early engineering result, not Khor Kalba field suitability or ecological benefit. No tank test is claimed or scheduled.

For accuracy, first establish reusable independent sufficiently detailed near-date evidence, then draw the frozen held-out probability sample and perform blind interpretation. For ecology, first establish current need and suitability, with no action/protection/natural regeneration as genuine alternatives; any intervention requires the separate outcome protocol (repository path: `VALIDATION_AND_BENEFIT_TESTS.md`). The three evidence tracks cannot substitute for each other.

## Reproducible companion and status

Connected concept Colab notebook (repository path: `../supplementary/Mangrove_Pulse_Connected_Concept.ipynb`) embeds the unchanged saved requests, this note, exact screening code/tests and blank parameter records. It needs no dataset upload, package installation or accelerator, and exports a downloadable assessment ZIP when run in Colab. Source is concept_screen.py (repository path: `../src/concept_screen.py`). Default engineering inputs are unknown; saved energy/time values remain explicitly hypothetical. It never returns deployment authorization, independent accuracy, superiority or ecological-benefit proof. Synthetic tests test equations and record handling only.

Source prepared; new Colab execution is not yet verified. The frozen EO inputs, threshold, split, predictions, scores and actual previously executed notebooks are unchanged. No new field data, local scientific run, PDF, publication or submission.


In [ ]:
# Engineering inputs deliberately start unknown. Do not silently substitute favourable values.
# Edit records only with units, evidence kind, source and conditions/uncertainty.
# Declared measurements are not independently verified by this software.
parameters = module.blank_parameters()

# Copied rounded illustrative outputs from the previously completed hypothetical scenario.
# These values are NOT a measured Khor Kalba mission and do not populate aircraft mass/geometry.
# The old scenario assumes zero extra overhead; complete measured mission costs remain unknown.
saved_example = {
    'hover_complete_energy_wh': 170.7,
    'supported_complete_energy_wh': 128.0,
    'hover_complete_time_s': 750.0,
    'supported_complete_time_s': 840.0,
}
for key, value in saved_example.items():
    parameters[key].update(value=value, kind='saved_hypothetical_result',
        source='examples/inspection/energy_scenario_excerpt.json; MP-DRONE-ENERGY-01',
        conditions_and_uncertainty='Rounded hypothetical scenario; equal attempted tasks, assumed loads/transition times, zero extra overhead; no measured acceptance fractions')

assessment = module.assess(parameters)
print(json.dumps(assessment, indent=2))
assert assessment['deployment_authorized'] is False
assert all(value == 'unproven' for value in assessment['claims'].values())
print('Unknowns are evidence gaps, not zero costs or passed feasibility tests.')


## 3. What to test next

Historical accuracy needs qualified independent near-date reference evidence; current drone visits cannot substitute. Platform performance needs a useful observation with a defined acceptance rule and a simpler matched comparator, followed by tests of sensor bias, whole-mission resources, transitions and recovery in permitted controlled conditions. Ecological benefit requires evidence of current need and a separate controlled outcome study.

The default calculation uses copied hypothetical resource values and synthetic equation checks, leaving physical task parameters unknown. It generates no new satellite processing, aircraft simulation, mission, field label or ecological outcome. Flight, water contact and planting require separate authorization. The final ZIP retains source, input records and a manifest for auditing assumptions.

In [ ]:
from pathlib import Path
import tempfile
import zipfile

export_root = Path(tempfile.mkdtemp(prefix='mangrove_connected_'))
export_dir = export_root / 'assessment'
export_dir.mkdir()
status = {
    'experiment_id': 'MP-CONNECTED-CONCEPT-01',
    'evidence_kind': 'conditional_analytic_assessment_and_synthetic_software_tests',
    'synthetic_tests_run': result.testsRun,
    'synthetic_tests_passed': result.wasSuccessful(),
    'actual_drone_missions': 0,
    'independent_2018_labels': 0,
    'ecological_outcome_observations': 0,
    'deployment_authorized': False,
    'claims': assessment['claims'],
    'requests_sha256': REQUESTS_SHA256,
}
payloads = {
    'inspection_requests.json': REQUESTS_TEXT,
    'connected_concept_assessment.md': NOTE_TEXT,
    'concept_screen.py': SCREEN_SOURCE,
    'test_concept_screen.py': TEST_SOURCE,
    'parameter_records.json': json.dumps(parameters, indent=2) + '\n',
    'conditional_assessment.json': json.dumps(assessment, indent=2) + '\n',
    'status.json': json.dumps(status, indent=2) + '\n',
}
for name, content in payloads.items():
    (export_dir / name).write_bytes(content.encode('utf-8'))
manifest = {'algorithm': 'sha256', 'files': [
    {'path': name, 'bytes': len(content.encode('utf-8')), 'sha256': hashlib.sha256(content.encode('utf-8')).hexdigest()}
    for name, content in sorted(payloads.items())]}
(export_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
zip_path = export_root / 'Mangrove_Pulse_Connected_Concept.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(export_dir.iterdir()):
        archive.write(path, arcname=path.name)
print(json.dumps(status, indent=2))
print('ZIP:', zip_path.name, 'bytes:', zip_path.stat().st_size, 'SHA-256:', hashlib.sha256(zip_path.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Download fallback:', zip_path)
